<a href="https://colab.research.google.com/github/johnmichaels4867-create/IS4487/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# We will try the expected GitHub paths, with a reliable fallback to the official TidyTuesday source of this exact dataset.
urls = [
    'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv',
    'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/Datasets/hotels.csv',
    'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/IDATA/hotels.csv'
]

df = None
for url in urls:
    try:
        df = pd.read_csv(url)
        print(f"Dataset successfully loaded from: {url}")
        break
    except Exception as e:
        continue

if df is None:
    print("Error: Could not load the dataset from any of the provided URLs.")
else:
    # Display basic information
    print(f"Initial shape: {df.shape}")
    print(df.info())

    # 1. Clean and Prepare Data
    # Drop high cardinality or irrelevant identifier columns
    cols_to_drop = ['agent', 'company', 'reservation_status_date', 'reservation_status']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')

    # Handle missing values
    # Fill categorical missing values with 'Unknown'
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown')

    # Fill numerical missing values with median
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    # Encode categorical variables using Label Encoding
    label_encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

    # 2. Split features and target
    X = df.drop(columns=['is_canceled'])
    y = df['is_canceled']

    # 3. Train-test split (70/30) before any scaling/transformation
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

    print(f"\nTraining set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")
    display(X_train.head())

Dataset successfully loaded from: https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv
Initial shape: (119390, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 

,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
9710,1,122,2016,2,53,27,0,4,2,0.0,...,0,3,3,3,0,0,2,74.75,0,2
40307,0,20,2015,5,30,22,0,2,2,0.0,...,0,0,0,0,0,0,3,62.00,0,0
110051,0,87,2017,0,16,16,2,3,3,0.0,...,0,3,3,0,0,0,2,171.00,0,0
22999,1,0,2017,3,5,2,0,1,1,0.0,...,1,0,0,0,0,0,2,43.00,0,1
8344,1,42,2016,11,39,23,0,1,2,0.0,...,0,0,0,0,0,0,2,109.00,0,1


### ✍️ Your Response: 🔧
1. **Total Rows and Columns:** The initial dataset contains 119,390 rows and 32 columns.

2. **Feature Types:** The dataset contains a mix of 12 categorical variables and 20 numerical variables.

3. **Cleaning and Preparation Steps:** We dropped non-predictive columns, filled missing values with default labels or medians, encoded all categories numerically, and then split the data into a 70% training and 30% testing partition.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [6]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and train the Gaussian Naïve Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict on the test set
y_pred_nb = nb_model.predict(X_test)

# Evaluation Metrics
print("=== Naïve Bayes Model Evaluation ===\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_nb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

=== Naïve Bayes Model Evaluation ===

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.37      0.52     22478
           1       0.46      0.91      0.61     13339

    accuracy                           0.57     35817
   macro avg       0.67      0.64      0.57     35817
weighted avg       0.72      0.57      0.55     35817

Confusion Matrix:
[[ 8331 14147]
 [ 1214 12125]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. **Model Performance and Metric:** The model achieved an overall accuracy of 57%, but given the high cost of unexpected cancellations, **Recall** for the canceled class (which is 91%) is the best metric to judge performance because it ensures we flag almost all potential cancellations.

2. **Business Utility:** This model is useful as an automated real-time flagging tool to alert staff of high-risk bookings during reservation, allowing them to proactively confirm bookings or send payment reminders.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Train a Support Vector Machine classifier
# We use LinearSVC as it is highly optimized for linear classification and much faster than SVC(kernel='linear') on large datasets
svm_model = LinearSVC(random_state=42, max_iter=2000)
svm_model.fit(X_train_scaled, y_train)

# 3. Make predictions
y_pred_svm = svm_model.predict(X_test_scaled)

# 4. Evaluation Metrics
print("=== Support Vector Machine (SVM) Model Evaluation ===\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_svm))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

=== Support Vector Machine (SVM) Model Evaluation ===

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.93      0.85     22478
           1       0.82      0.56      0.67     13339

    accuracy                           0.79     35817
   macro avg       0.80      0.75      0.76     35817
weighted avg       0.80      0.79      0.78     35817

Confusion Matrix:
[[20876  1602]
 [ 5815  7524]]


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. **Model Performance and Metric:** The linear SVM model performs significantly better overall, achieving an overall accuracy of 79% (compared to 57% for Na%ve Bayes). To judge the performance, **Weighted F1-score** (0.78) or **Accuracy** are the best metrics here, as they represent a more balanced trade-off between identifying cancellations (Recall of 56%) and maintaining high precision (82%).

2. **Business Situations for SVM:** SVM is highly beneficial in situations with complex, high-dimensional customer behaviors where relationships between variables are non-linear or multi-faceted, such as separating high-value corporate group booking behaviors from individual leisure travelers.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [8]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize MLPClassifier with a simple 2-layer architecture (32, 16 neurons)
# We set early_stopping=True and validation_fraction=0.1 to use a true validation split from the training data.
mlp_model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
    max_iter=500
)

# 2. Train the model on the scaled training set
mlp_model.fit(X_train_scaled, y_train)

# 3. Make predictions on the scaled test set
y_pred_mlp = mlp_model.predict(X_test_scaled)

# 4. Evaluation Metrics
print("=== Neural Network (MLPClassifier) Model Evaluation ===\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_mlp))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

=== Neural Network (MLPClassifier) Model Evaluation ===

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.92      0.89     22478
           1       0.84      0.74      0.79     13339

    accuracy                           0.85     35817
   macro avg       0.85      0.83      0.84     35817
weighted avg       0.85      0.85      0.85     35817

Confusion Matrix:
[[20614  1864]
 [ 3439  9900]]


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. **Model Comparison:** This Neural Network outperformed both Na%ve Bayes (57% accuracy) and Linear SVM (79% accuracy), achieving an overall accuracy of 85% with a highly balanced 74% recall and 84% precision for predicting cancellations.

2. **Black Box Comfort Level:** The business might have initial reservations because neural networks do not provide easily interpretable feature importances; however, the highly superior predictive accuracy (85%) and minimal error rates would likely justify its operational use for automated email reminders.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [9]:
from sklearn.metrics import accuracy_score

# Calculate accuracies
nb_acc = accuracy_score(y_test, y_pred_nb)
svm_acc = accuracy_score(y_test, y_pred_svm)
mlp_acc = accuracy_score(y_test, y_pred_mlp)

# Print side-by-side comparison
print("=== Model Accuracy Comparison ===")
print(f"Na%ve Bayes Accuracy:       {nb_acc * 100:.2f}%")
print(f"Support Vector Machine:     {svm_acc * 100:.2f}%")
print(f"Neural Network (MLP):       {mlp_acc * 100:.2f}%\n")

# Summarize the best-performing model
model_accuracies = {"Na%ve Bayes": nb_acc, "Support Vector Machine": svm_acc, "Neural Network": mlp_acc}
best_model = max(model_accuracies, key=model_accuracies.get)

print("=== Performance Summary ===")
print(f"The best performing model in terms of overall accuracy is the {best_model} with {model_accuracies[best_model] * 100:.2f}% accuracy.")

=== Model Accuracy Comparison ===
Na%ve Bayes Accuracy:       57.11%
Support Vector Machine:     79.29%
Neural Network (MLP):       85.19%

=== Performance Summary ===
The best performing model in terms of overall accuracy is the Neural Network with 85.19% accuracy.


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. **Model Comparison across Criteria:**
   - **Best Overall Accuracy:** Neural Network (MLP) (85.19% vs. SVM's 79.29% and Na%ve Bayes' 57.11%).
   - **Best Training Time:** Na%ve Bayes (instantaneous), whereas Neural Networks and SVM require scaling and iterations.
   - **Best Interpretability:** Na%ve Bayes (based on simple conditional probabilities), while Neural Networks are a complete "black box".
   - **Ease of Use:** Na%ve Bayes (requires no feature scaling), whereas SVM and Neural Network require strict feature preprocessing.

2. **Deployment Recommendation:** I recommend deploying the **Neural Network** model. With its high accuracy (85.19%) and highly balanced precision/recall, it minimizes costly false-negative errors, allowing hotel managers to proactively manage staff and food inventory with confidence.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. **Executive Recommendation:**
I recommend implementing the Neural Network model, which delivers 85.19% predictive accuracy. This model directly addresses unpredictable booking cancellations, enabling managers to proactively adjust staff schedules, optimize room inventories, and secure revenue. While the model is highly accurate, its "black box" nature makes it harder to interpret directly. To mitigate this risk and further improve performance, future iterations should incorporate external data such as real-time local weather forecasts, flight cancellations, and seasonal regional events.

2. **Connection to Learning Outcomes:**
This analysis connects directly to my learning outcomes by demonstrating how to preprocess raw business data, handle target leakage, implement multiple modern scikit-learn classifiers, and translate complex model metrics (like precision and recall) into actionable business strategy.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"